# Session XI: Python I Foundations Review

Reviews the **core concepts** from Sessions I–IX, the same building blocks you combined in the Session X Budget Manager lab.

---

## Exercise 1: Type Conversion & Error Handling

Write `safe_convert(values)` that takes a mixed list, tries `float()` on each, returns `(valid_list, error_count)`.

### How to approach this code

The input list mixes numbers, convertible strings, invalid strings, and `None`, so the function must continue after an individual conversion error. Catch `ValueError` and `TypeError`, keep valid floats, and count failures separately. Python also converts `True` to `1.0`; decide whether that is appropriate for this data.


In [ ]:
import math


def safe_convert(values: list) -> tuple[list[float], int]:
    valid = []
    errors = 0
    for value in values:
        # float(True) gives 1.0, but a boolean is not a measurement, so count it as an error
        if isinstance(value, bool):
            errors += 1
            continue
        try:
            number = float(value)
        except (ValueError, TypeError):  # ValueError: "hello", "" / TypeError: None
            errors += 1
            continue
        if math.isfinite(number):  # float("nan") and float("inf") succeed but are not real data
            valid.append(number)
        else:
            errors += 1
    return valid, errors

test = [10, "3.14", None, "hello", 0, "42", True, "", 7.5]
valid, errors = safe_convert(test)
print(f"Valid: {valid}")
print(f"Errors: {errors}")

---
## Exercise 2: Data Structures & Comprehensions

Given students data, write:
- `get_honor_roll(students)` — names with avg >= 90
- `get_unique_courses(students)` — set of all courses
- `average_by_course(students)` — dict of course averages

### How to approach this code

Each student record contains a nested course-to-score dictionary. The honor roll is an ordered list, unique course names belong in a set, and average-by-course results fit a dictionary. The examples show how the question determines the data structure.


In [ ]:
students = [
    {"name": "Alice", "courses": {"Math": 95, "Physics": 88, "CS": 92}},
    {"name": "Bob", "courses": {"Math": 78, "Chemistry": 85, "CS": 90}},
    {"name": "Carol", "courses": {"Physics": 92, "CS": 95, "Math": 91}},
    {"name": "David", "courses": {"Chemistry": 70, "Math": 65, "Biology": 80}},
    {"name": "Eve", "courses": {"CS": 98, "Math": 94, "Physics": 96}},
]


def student_average(student: dict) -> float:
    scores = student["courses"].values()
    return sum(scores) / len(scores)


def get_honor_roll(students: list[dict]) -> list[str]:
    return [s["name"] for s in students if student_average(s) >= 90]


def get_unique_courses(students: list[dict]) -> set[str]:
    return {course for s in students for course in s["courses"]}


def average_by_course(students: list[dict]) -> dict[str, float]:
    scores_by_course = {}
    for s in students:
        for course, score in s["courses"].items():
            scores_by_course.setdefault(course, []).append(score)
    return {course: round(sum(scores) / len(scores), 2)
            for course, scores in sorted(scores_by_course.items())}


print("Honor roll:", get_honor_roll(students))
print("Courses:", get_unique_courses(students))
print("Averages:", average_by_course(students))

---
## Exercise 3: File I/O & String Cleaning

Parse messy CSV, clean names (title case), validate emails with regex, validate ages as int. Return `(valid_records, error_messages)`.

### How to approach this code

The CSV sample contains extra whitespace, inconsistent capitalization, a bad age, and malformed records. Clean each field before validating it; collect rejected rows with messages instead of letting one bad row stop the entire import. Regex tests email shape but cannot prove deliverability.


In [ ]:
import csv
import io
import re

messy = """name,email,age
   alice SMITH , ALICE@EMAIL.COM , 25
 bob Jones,bob@email.com,thirty
  CAROL  white  , carol@email ,  30
 David Brown, david.brown@university.edu, 22
  , missing@email.com, 28
eve GREEN,eve.green@company.co.uk,27
"""

EMAIL_PATTERN = re.compile(r"^[\w.+-]+@[\w-]+(\.[\w-]+)*\.[a-z]{2,}$")


def clean_and_validate(raw: str) -> tuple[list[dict], list[str]]:
    valid = []
    errors = []
    reader = csv.reader(io.StringIO(raw.strip()))
    next(reader)  # skip the header row
    for line_number, row in enumerate(reader, start=2):
        if len(row) != 3:
            errors.append(f"Line {line_number}: expected 3 fields, got {len(row)}")
            continue
        name = " ".join(row[0].split()).title()  # collapse inner spaces, then title case
        email = row[1].strip().lower()
        age_text = row[2].strip()

        if not name:
            errors.append(f"Line {line_number}: name is missing")
            continue
        if not EMAIL_PATTERN.match(email):
            errors.append(f"Line {line_number}: invalid email {email!r}")
            continue
        try:
            age = int(age_text)
        except ValueError:
            errors.append(f"Line {line_number}: age {age_text!r} is not a whole number")
            continue
        if not 0 < age < 130:
            errors.append(f"Line {line_number}: age {age} is out of range")
            continue

        valid.append({"name": name, "email": email, "age": age})
    return valid, errors


valid, errors = clean_and_validate(messy)
print(f"Valid ({len(valid)}): {valid}")
print(f"Errors ({len(errors)}): {errors}")

---
## Exercise 4: OOP & Patterns

Build `Book` + `Library` classes. Library needs: add_book, search(keyword), checkout(title), available_books property, __len__, summary().

### How to approach this code

`Book` models one title and its checkout state; `Library` manages a collection of books. Search should not mutate the collection, while checkout changes availability only for a matching available book. Properties and special methods make the domain objects easier to inspect and use.


In [ ]:
class Book:
    def __init__(self, title: str, author: str, year: int, is_available: bool = True) -> None:
        self.title = title
        self.author = author
        self.year = year
        self.is_available = is_available

    def __repr__(self) -> str:
        status = "available" if self.is_available else "checked out"
        return f"Book({self.title!r} by {self.author}, {self.year}, {status})"


class Library:
    def __init__(self) -> None:
        self.books: list[Book] = []

    def add_book(self, book: Book) -> None:
        self.books.append(book)

    def search(self, keyword: str) -> list[Book]:
        keyword = keyword.lower()
        return [b for b in self.books
                if keyword in b.title.lower() or keyword in b.author.lower()]

    def checkout(self, title: str) -> bool:
        for book in self.books:
            if book.title.lower() == title.lower() and book.is_available:
                book.is_available = False
                return True
        return False  # not found or already checked out

    @property
    def available_books(self) -> list[Book]:
        return [b for b in self.books if b.is_available]

    def __len__(self) -> int:
        return len(self.books)

    def summary(self) -> None:
        print(f"Library: {len(self)} books, {len(self.available_books)} available")
        for book in sorted(self.books, key=lambda b: b.year):
            print(f"  {book}")


lib = Library()
lib.add_book(Book("Python Crash Course", "Eric Matthes", 2019))
lib.add_book(Book("Clean Code", "Robert Martin", 2008))
lib.add_book(Book("Python Cookbook", "David Beazley", 2013))
print(f"Total: {len(lib)}")
print(f"Search 'python': {lib.search('python')}")
print(f"Checkout 'Clean Code': {lib.checkout('Clean Code')}")
print(f"Checkout again: {lib.checkout('Clean Code')}")
lib.summary()

---
## Exercise 5: Integrated project

Build a Contact Manager with: Contact class (name, email, phone, category), validate_email/phone with regex, JSON save/load, search with comprehensions, report grouped by category.

### How to approach this code

The contact manager combines a data model, static validation methods, JSON serialization, search, and category reports. The object keeps related fields together, validation protects the input boundary, and conversion methods allow records to be saved and reconstructed. Validate before persisting so bad data does not become permanent.


In [ ]:
import json
import os
import re


class Contact:
    VALID_CATEGORIES = ["Family", "Friend", "Work", "Other"]

    def __init__(self, name: str, email: str, phone: str, category: str = "Other") -> None:
        name = name.strip()
        if not name:
            raise ValueError("Name cannot be empty")
        if not Contact.validate_email(email):
            raise ValueError(f"Invalid email: {email!r}")
        if not Contact.validate_phone(phone):
            raise ValueError(f"Invalid phone: {phone!r}")
        if category not in Contact.VALID_CATEGORIES:
            raise ValueError(f"Category must be one of {Contact.VALID_CATEGORIES}")
        self.name = name
        self.email = email.strip().lower()
        self.phone = re.sub(r"[\s-]", "", phone)  # store digits without spaces or dashes
        self.category = category

    @staticmethod
    def validate_email(email: str) -> bool:
        return re.fullmatch(r"[\w.+-]+@[\w-]+(\.[\w-]+)*\.[A-Za-z]{2,}", email.strip()) is not None

    @staticmethod
    def validate_phone(phone: str) -> bool:
        # optional +country code, then 9 digits; spaces and dashes allowed
        digits = re.sub(r"[\s-]", "", phone)
        return re.fullmatch(r"(\+\d{1,3})?\d{9}", digits) is not None

    def to_dict(self) -> dict:
        return {"name": self.name, "email": self.email,
                "phone": self.phone, "category": self.category}

    @classmethod
    def from_dict(cls, data: dict) -> "Contact":
        return cls(data["name"], data["email"], data["phone"], data.get("category", "Other"))

    def __repr__(self) -> str:
        return f"Contact({self.name!r}, {self.email!r}, {self.phone!r}, {self.category!r})"


def save_contacts(contacts: list[Contact], filepath: str = "data/contacts.json") -> None:
    folder = os.path.dirname(filepath)
    if folder:
        os.makedirs(folder, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump([c.to_dict() for c in contacts], f, indent=2)


def load_contacts(filepath: str = "data/contacts.json") -> list[Contact]:
    try:
        with open(filepath, encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        return []
    except json.JSONDecodeError as error:
        print(f"Could not read {filepath}: {error}")
        return []
    if not isinstance(data, list):
        print(f"Could not read {filepath}: expected a list of contacts")
        return []
    contacts = []
    for item in data:
        try:
            contacts.append(Contact.from_dict(item))
        except (KeyError, TypeError, ValueError) as error:
            print(f"Skipping bad record {item!r}: {error}")
    return contacts


def search_contacts(contacts: list[Contact], keyword: str) -> list[Contact]:
    keyword = keyword.lower()
    return [c for c in contacts
            if keyword in c.name.lower() or keyword in c.email or keyword in c.phone]


def report_by_category(contacts: list[Contact]) -> None:
    for category in Contact.VALID_CATEGORIES:
        group = [c for c in contacts if c.category == category]
        if group:
            print(f"{category} ({len(group)}):")
            for c in sorted(group, key=lambda c: c.name):
                print(f"  {c.name:<8} {c.email:<22} {c.phone}")


contacts = [
    Contact("Alice", "alice@email.com", "612345678", "Friend"),
    Contact("Bob", "Bob@Work.com", "+34 612-345-679", "Work"),
    Contact("Carol", "carol@family.org", "698 765 432", "Family"),
    Contact("Dan", "dan@work.com", "611222333", "Work"),
]

try:
    Contact("Bad", "not-an-email", "123", "Friend")
except ValueError as error:
    print(f"Rejected: {error}")

save_contacts(contacts, "data/contacts.json")
loaded = load_contacts("data/contacts.json")
print(f"Loaded {len(loaded)} contacts")
print("Search 'work':", search_contacts(loaded, "work"))
report_by_category(loaded)

---
## Exercise 6: Review & Debugging Mini Lab
**Objective:** Practice reading code, spotting weak points, and improving readability.

Create a function called `review_code_snippet(snippet)` that:
- counts how many `if` statements appear
- flags repeated code patterns using a simple keyword search
- returns a small report dictionary with `if_count`, `repeated_patterns`, and `status`

This is intentionally a review exercise. The main goal is to think like a code reviewer.

### How to approach this code

The review helper treats source code as text and reports simple signals such as repeated calls or conditionals. This is a teaching heuristic rather than a static analyzer: counting text can be fooled by comments and strings. The structured result can guide a human review but cannot certify code quality.


In [ ]:
import re


def review_code_snippet(snippet: str) -> dict:
    """Review a short code string and report likely quality issues."""
    # count `if` and `elif` at the start of a line (so "if" inside a word or string is ignored)
    if_count = len(re.findall(r"^\s*(?:el)?if\b", snippet, flags=re.MULTILINE))

    # \b stops "int(" from matching inside "print("
    counts = {f"{name}(": len(re.findall(rf"\b{name}\(", snippet))
              for name in ["print", "input", "open", "int", "float"]}
    repeated_patterns = {call: n for call, n in counts.items() if n > 1}

    issues = []
    if if_count > 3:
        issues.append("many if statements: consider a dict lookup or a helper function")
    if repeated_patterns:
        issues.append("repeated calls: consider a loop or a function")
    status = "needs review" if issues else "ok"

    return {
        "if_count": if_count,
        "repeated_patterns": repeated_patterns,
        "status": status,
        "suggestions": issues,
    }


report = review_code_snippet("if x > 0:\n    print(x)\nif x < 0:\n    print(-x)")
print(report)
print(review_code_snippet("total = sum(values)\nprint(total)"))